# 02 - Yeast Proteome Screening on Colab

Runs the full `crypticip` screening workflow on the yeast AlphaFold
proteome (UP000002311). This is **resource-heavy** — read the
limitations below before running.

## Resource warnings
- The yeast AlphaFold tar is **~3 GB compressed, ~5-6 GB extracted**
  (~6000 PDB files). Colab free-tier provides ~80 GB ephemeral disk,
  so this fits — but the runtime can be reclaimed at any time.
- Human (UP000005640) is ~80 GB and will NOT fit on free-tier Colab.
- Without fpocket / APBS installed, screening completes plumbing but
  produces zero pockets. Use the `INSTALL_EXTERNAL=True` section
  from `01_validation_colab.ipynb` first.
- **Strongly recommended:** use `--limit` for a first run.


## 1. Clone + install

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
WORK = pathlib.Path('/content/cryptic-ip-pipeline')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
!pip install -q -e .


## 2. (Optional) Mount Drive

Putting outputs on Drive makes them survive a runtime swap. Outputs
are a few MB (CSVs, JSON, MD) — easily fits Drive free tier.

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    import datetime, os, pathlib, shutil
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = pathlib.Path('/content/drive/MyDrive/crypticip_results')
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    # Symlink results dir to Drive so CLI writes persist.
    # Carefully handle pre-existing `results` from the cloned repo:
    target = pathlib.Path('results')
    if target.is_symlink():
        # If it already points to DRIVE_RESULTS, leave it alone.
        try:
            current = pathlib.Path(os.readlink(target)).resolve()
        except OSError:
            current = None
        if current == DRIVE_RESULTS.resolve():
            print('results/ already symlinked to Drive — leaving as-is')
        else:
            target.unlink()
            target.symlink_to(DRIVE_RESULTS)
            print('results/ -> Drive (replaced stale symlink)')
    elif target.exists():
        # Preserve any local data by moving to a timestamped backup.
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        backup = target.with_name(f'results.local_backup_{ts}')
        shutil.move(str(target), str(backup))
        print(f'moved existing results/ -> {backup}')
        target.symlink_to(DRIVE_RESULTS)
        print('results/ -> Drive')
    else:
        target.symlink_to(DRIVE_RESULTS)
        print('results/ -> Drive')


## 3. Verify env

In [ ]:
!crypticip check-env

## 4. Download the yeast proteome

Tip: keep an eye on disk usage with `!df -h /content`. Use
`--limit N` to extract only the first N PDB files for a first run.

In [ ]:
# Smoke variant — only extract 50 files
!crypticip download-proteome --organism yeast --limit 50

In [ ]:
# Full download (commented out by default — uncomment when ready):
# !crypticip download-proteome --organism yeast --resume

## 5. Build the index

In [ ]:
!crypticip index-proteome --organism yeast

## 6. Small fast screen first

Always start with `--limit` and `--workers 2` to time things on
your Colab runtime. Then scale up.

In [ ]:
!crypticip screen --organism yeast --mode fast --workers 2 --limit 25

## 7. Full screen (resumable)

Uncomment when ready. `--resume` (the default) makes re-runs idempotent —
if Colab disconnects, just re-execute the cell.

In [ ]:
# !crypticip screen --organism yeast --mode fast --workers 4 --resume
# For deeper electrostatics (APBS, slower):
# !crypticip screen --organism yeast --mode full --workers 4 --resume


## 8. Report

In [ ]:
!crypticip report --organism yeast

## 9. PyMOL sessions for top hits

In [ ]:
# .pml session files for the top 20 candidates.
# Use --render only if PyMOL was installed (section 2 of 01_validation_colab).
!crypticip pymol --organism yeast --top 20

## 10. Experimental follow-up

In [ ]:
!crypticip experimental-plan --organism yeast --top 25

## 11. Persist outputs

If you mounted Drive in section 2, outputs are already on Drive.
Otherwise zip & download:

In [ ]:
import shutil, pathlib
out = shutil.make_archive('/content/yeast_results', 'zip', 'results')
print('zipped to:', out, 'size MB:', round(pathlib.Path(out).stat().st_size/1e6, 2))
try:
    from google.colab import files
    files.download(out)
except Exception as e:
    print('non-Colab env, skip files.download:', e)


## 12. Tips for production runs

- A real screen on a workstation (16 cores, full toolchain) takes
  ~2-4 hours on yeast.
- For human (~20k structures): plan for ~24-72 h on a multi-core
  workstation and ~100 GB disk. Colab is **not** the right venue.
- The repo does **not** commit any proteome data or per-structure
  outputs. Keep them on Drive or local disk.
